# 00 · Getting acquainted: event data and a first look at confidence

**Goal of this notebook:** get comfortable with soccer event data, then run one small, honest experiment on the project question: *does a player's behaviour change after something that should boost his confidence, and which parts of his game move most?*

Roadmap

1. What's in the StatsBomb open data
2. One match, opened up: what an "event" is
3. A small `prep()` helper (clock + coordinates)
4. Seeing it on a pitch
5. From events to *attributes* (choices vs. execution)
6. First experiment: what does a shooter do after scoring vs. after missing?
7. What this does and doesn't show, and where to take it

Data: [StatsBomb open data](https://github.com/statsbomb/open-data), loaded with `statsbombpy`. If you publish anything from it, StatsBomb asks to be credited as the source.

In [ ]:
# If you haven't installed these yet, uncomment and run once:
# %pip install statsbombpy mplsoccer pandas numpy matplotlib

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mplsoccer import Pitch
from statsbombpy import sb

# statsbombpy warns on every call that you're on the free tier. That's expected.
warnings.filterwarnings("ignore", message="credentials were not supplied")
pd.set_option("display.max_columns", 60)

CACHE = Path("../data/raw/statsbomb_notebook")   # downloaded matches get cached here
CACHE.mkdir(parents=True, exist_ok=True)

## 1 · What's in the open data

The data is organised as **competition → season → match → events**. `sb.competitions()` lists every competition-season that's free.

In [ ]:
comps = sb.competitions()

(comps.groupby(["competition_name", "competition_gender"])["season_name"]
      .agg(lambda s: ", ".join(sorted(s)))
      .to_frame("seasons"))

Two things to notice in that table:

- **Tournaments** (World Cups, Euros, Copa América) are complete but short: a player gets 3 to 7 matches.
- A few **full league seasons** are in there too. Those matter for this project, because confidence is a within-player, over-time idea and you need many matches per player to see it.

We'll start with the 2022 men's World Cup because it's small (64 matches) and you know the games. IDs are looked up by name rather than hard-coded.

In [ ]:
def find_season(competition_name, season_name):
    hit = comps[(comps["competition_name"] == competition_name) & (comps["season_name"] == season_name)]
    if len(hit) != 1:
        raise ValueError(
            f"Expected exactly one row for {competition_name!r} {season_name!r}, found {len(hit)}. "
            "Check the table above for the exact spelling."
        )
    return int(hit["competition_id"].iloc[0]), int(hit["season_id"].iloc[0])


COMP_ID, SEASON_ID = find_season("FIFA World Cup", "2022")

matches = (sb.matches(competition_id=COMP_ID, season_id=SEASON_ID)
             .sort_values(["match_date", "kick_off"])
             .reset_index(drop=True))

print(len(matches), "matches")
matches[["match_id", "match_date", "competition_stage", "home_team", "away_team", "home_score", "away_score"]].tail(8)

## 2 · One match, opened up

Let's take the final. `sb.events()` returns one row per **event**: every pass, carry, shot, pressure, duel, substitution and so on, in order.

In [ ]:
final = matches[matches["competition_stage"] == "Final"]
MATCH_ID = int(final["match_id"].iloc[0]) if len(final) else int(matches["match_id"].iloc[-1])

raw = sb.events(match_id=MATCH_ID)
print(raw.shape)
raw["type"].value_counts()

In [ ]:
cols = ["index", "period", "timestamp", "minute", "second", "team", "player", "position", "type",
        "location", "under_pressure", "pass_recipient", "pass_end_location", "pass_outcome",
        "shot_statsbomb_xg", "shot_outcome"]

raw.sort_values("index")[[c for c in cols if c in raw.columns]].iloc[200:215]

**How to read a row**

- `type` is what happened. Columns prefixed `pass_`, `shot_`, `carry_`, `dribble_` … only apply to that event type and are `NaN` everywhere else. That's why the table is ~100 columns wide and mostly empty.
- `location` is `[x, y]` on a **120 × 80** pitch. Coordinates are always from the point of view of the team doing the action, **attacking left → right**. So `x = 120` is the goal that team is attacking, for both teams, in both halves.
- `pass_outcome` is `NaN` when the pass was **completed**. Only failures get a label. (This catches everyone once.)
- `under_pressure` is `True` or `NaN`.
- `shot_statsbomb_xg` is the model's probability that the shot becomes a goal, given where and how it was taken.
- `timestamp` restarts at zero each period. `period` 5 is a penalty shootout.
- Rows come back **unsorted**; sort by `index`.
- `Ball Receipt*` (yes, with the asterisk) is logged for the player receiving a pass. It's a handy proxy for "showing for the ball".

## 3 · A small `prep()` helper

Three chores you'll do for every match: sort, build a continuous match clock, and unpack the coordinate lists into plain columns.

In [ ]:
NEEDED = ["pass_length", "pass_outcome", "pass_end_location", "carry_end_location", "shot_end_location",
          "shot_statsbomb_xg", "shot_outcome", "shot_type", "dribble_outcome", "under_pressure",
          "tactics", "substitution_replacement"]


def _coord(series, i):
    # pull coordinate i out of a column of [x, y] lists; NaN where there is no location
    return np.array([v[i] if isinstance(v, (list, tuple, np.ndarray)) else np.nan for v in series], dtype=float)


def prep(ev):
    # sort, drop shootouts, add `clock` (continuous minutes) and x / y / end_x / end_y
    ev = ev[ev["period"] <= 4].sort_values("index").reset_index(drop=True).copy()
    for c in NEEDED:                       # some columns are absent if the event never happened in a match
        if c not in ev.columns:
            ev[c] = np.nan

    secs = pd.to_timedelta(ev["timestamp"]).dt.total_seconds()
    offset = secs.groupby(ev["period"]).max().cumsum().shift(fill_value=0)   # length of the periods before this one
    ev["clock"] = (secs + ev["period"].map(offset)) / 60

    ev["x"], ev["y"] = _coord(ev["location"], 0), _coord(ev["location"], 1)
    ev["end_x"], ev["end_y"] = np.nan, np.nan
    for c in ["pass_end_location", "carry_end_location", "shot_end_location"]:
        ev["end_x"] = ev["end_x"].fillna(pd.Series(_coord(ev[c], 0), index=ev.index))
        ev["end_y"] = ev["end_y"].fillna(pd.Series(_coord(ev[c], 1), index=ev.index))
    return ev


ev = prep(raw)
ev[["clock", "team", "player", "type", "x", "y", "end_x", "end_y"]].iloc[200:206]

## 4 · Seeing it on a pitch

`mplsoccer` draws pitches in each provider's coordinate system. First the shot map: marker size is xG, filled markers are goals. Each team is drawn attacking to the right.

In [ ]:
pitch = Pitch(pitch_type="statsbomb", line_color="grey")

shots = ev[ev["type"] == "Shot"]
fig, axes = pitch.draw(ncols=2, figsize=(14, 5))

for ax, (team, s) in zip(axes, shots.groupby("team")):
    goal = s["shot_outcome"] == "Goal"
    size = s["shot_statsbomb_xg"] * 700 + 30
    pitch.scatter(s.loc[~goal, "x"], s.loc[~goal, "y"], s=size[~goal],
                  facecolors="none", edgecolors="tab:blue", ax=ax, label="no goal")
    pitch.scatter(s.loc[goal, "x"], s.loc[goal, "y"], s=size[goal],
                  color="tab:red", ax=ax, label="goal")
    ax.set_title(f"{team}: {len(s)} shots · {s['shot_statsbomb_xg'].sum():.2f} xG · {goal.sum()} goals")

axes[0].legend(loc="lower left")
plt.show()

Now one player's passes. I pick whoever took the most shots so the cell runs without knowing StatsBomb's spelling of names (they use full legal names, so check `ev["player"].unique()` before filtering by hand).

In [ ]:
PLAYER = shots["player"].value_counts().index[0]

p = ev[(ev["player"] == PLAYER) & (ev["type"] == "Pass")]
done = p["pass_outcome"].isna()

fig, ax = pitch.draw(figsize=(9, 6))
pitch.arrows(p.loc[done, "x"], p.loc[done, "y"], p.loc[done, "end_x"], p.loc[done, "end_y"],
             width=1.5, color="tab:blue", ax=ax, label="complete")
pitch.arrows(p.loc[~done, "x"], p.loc[~done, "y"], p.loc[~done, "end_x"], p.loc[~done, "end_y"],
             width=1.5, color="tab:red", alpha=0.7, ax=ax, label="incomplete")
ax.set_title(f"{PLAYER}: {len(p)} passes, {done.mean():.0%} completed")
ax.legend(loc="lower left")
plt.show()

## 5 · From events to attributes

The project asks *which attributes of a player's game* respond to confidence. Nobody hands you attributes; you build them out of events. A split that will be useful all the way through:

| | **Choice** (what he tries) | **Execution** (how well it comes off) |
|---|---|---|
| Shooting | shots taken, xG per shot, shot distance | goals minus xG |
| Passing | pass length, share played forward | completion % |
| Dribbling | take-ons attempted | take-on success % |
| Involvement | ball receipts, carries | n/a |
| Off the ball | pressures | n/a |

The working hypothesis for the whole project fits in one line: **confidence shows up in choices before it shows up in execution.** A confident player tries more and tries harder things; whether those come off is noisier and partly offsets (harder attempts fail more).

`window_stats()` turns any slice of events into these numbers. We'll reuse it for a whole match here and for 15-minute windows in section 6.

In [ ]:
def window_stats(df):
    # summarise a slice of events (one player, or a pooled group of players)
    passes = df[df["type"] == "Pass"]
    shots_ = df[df["type"] == "Shot"]
    drib = df[df["type"] == "Dribble"]
    return {
        # involvement
        "receipts": int((df["type"] == "Ball Receipt*").sum()),
        "passes": len(passes),
        "carries": int((df["type"] == "Carry").sum()),
        "pressures": int((df["type"] == "Pressure").sum()),
        # choices
        "shots": len(shots_),
        "dribbles": len(drib),
        "fwd_pass_share": ((passes["end_x"] - passes["x"]) > 0).mean() if len(passes) else np.nan,
        "pass_length": passes["pass_length"].mean() if len(passes) else np.nan,
        "xg_per_shot": shots_["shot_statsbomb_xg"].mean() if len(shots_) else np.nan,
        # execution
        "pass_cmp": passes["pass_outcome"].isna().mean() if len(passes) else np.nan,
        "dribble_win": (drib["dribble_outcome"] == "Complete").mean() if len(drib) else np.nan,
        "goals_minus_xg": (shots_["shot_outcome"] == "Goal").sum() - shots_["shot_statsbomb_xg"].sum(),
    }


def on_pitch(ev):
    # clock time each player came on and went off (red cards are ignored: fine for now, fix later)
    end = ev["clock"].max()
    span = {}
    for r in ev[ev["type"] == "Starting XI"].itertuples():
        if isinstance(r.tactics, dict):
            for slot in r.tactics["lineup"]:
                span[slot["player"]["name"]] = [0.0, end]
    for r in ev[ev["type"] == "Substitution"].itertuples():
        span.setdefault(r.player, [0.0, end])[1] = r.clock
        sub = r.substitution_replacement
        span[sub["name"] if isinstance(sub, dict) else sub] = [r.clock, end]
    return pd.DataFrame.from_dict(span, orient="index", columns=["on", "off"])


def player_match_table(ev):
    mins = on_pitch(ev)
    by_player = ev.dropna(subset=["player"]).groupby("player")
    out = pd.DataFrame({name: window_stats(g) for name, g in by_player}).T.astype(float)
    out.insert(0, "minutes", (mins["off"] - mins["on"]).reindex(out.index).round(1))
    out.insert(0, "team", by_player["team"].first())
    return out


player_match_table(ev).sort_values("shots", ascending=False).head(12).round(2)

Scan that table for a minute. Two things worth internalising before any modelling:

- **Position dominates everything.** A centre-back and a winger live in different rows of this table. Any confidence effect has to be measured *within* a player (him vs. his own baseline), never across players.
- **Single-match numbers are tiny.** Three shots, two dribbles. Rates built on these are mostly noise, which is why the next section pools across a whole tournament and why the real project needs full seasons.

## 6 · First experiment: after a goal vs. after a miss

Confidence can't be observed, so we look for a **shock** that should move it and watch what the player does next.

The shock here is scoring. The trick is the comparison group: every shot is a weighted coin flip with probability ≈ xG. So compare

- shooters whose shot **went in**, against
- shooters whose shot **didn't**,

and ask how each one's next 15 minutes differ from his previous 15. Both groups just took a shot, so both were similarly involved and in similar positions beforehand. What separates them is mostly the bounce of the ball.

One obvious confound: a goal changes the score, and the score changes how the *whole team* plays (and there's a minute of celebration with no football). So we also compute the same change for the shooter's **teammates** and subtract it. What's left is the part specific to the scorer.

First, load every match (a couple of minutes the first time; cached to disk after that).

In [ ]:
def load_events(match_id):
    f = CACHE / f"{match_id}.pkl"
    if f.exists():
        return pd.read_pickle(f)
    df = sb.events(match_id=match_id)
    df.to_pickle(f)
    return df


all_ev = {}
for i, mid in enumerate(matches["match_id"], 1):
    all_ev[int(mid)] = prep(load_events(int(mid)))
    print(f"\r{i}/{len(matches)} matches loaded", end="")

In [ ]:
W = 15   # window length in minutes on each side of the shot

COUNTS = ["receipts", "passes", "carries", "pressures", "shots", "dribbles"]   # volumes
RATES = ["fwd_pass_share", "pass_length", "pass_cmp"]                           # averages over passes
METRICS = COUNTS + RATES


def change_around(df, t):
    # after-minus-before change in each metric, for the events in df, around clock time t
    before = window_stats(df[(df["clock"] >= t - W) & (df["clock"] < t)])
    after = window_stats(df[(df["clock"] > t) & (df["clock"] <= t + W)])
    return {m: after[m] - before[m] for m in METRICS}


rows = []
for mid, m_ev in all_ev.items():
    mins = on_pitch(m_ev)
    open_shots = m_ev[(m_ev["type"] == "Shot") & (m_ev["shot_type"] != "Penalty")]
    for s in open_shots.itertuples():
        if s.player not in mins.index:
            continue
        on, off = mins.loc[s.player]
        if s.clock - W < on or s.clock + W > off:      # need a full window on both sides
            continue
        team = m_ev[m_ev["team"] == s.team]
        me = change_around(team[team["player"] == s.player], s.clock)
        mates = change_around(team[team["player"] != s.player], s.clock)

        row = {"match_id": mid, "player": s.player, "clock": s.clock,
               "xg": s.shot_statsbomb_xg, "goal": s.shot_outcome == "Goal"}
        for m in METRICS:
            row[m] = me[m]                                                    # shooter's own change
            row[m + "_adj"] = me[m] - (mates[m] / 10 if m in COUNTS else mates[m])   # minus the average teammate's
        rows.append(row)

exp = pd.DataFrame(rows)
print(f"{len(exp)} shots kept: {exp['goal'].sum()} goals, {(~exp['goal']).sum()} non-goals")
exp.head()

Each row is one shot. The metric columns hold the shooter's **change** (next 15 minutes minus previous 15). The `_adj` columns subtract the average teammate's change over the same windows.

Now the comparison: mean change after goals minus mean change after non-goals, for each metric. To put passes and pass-lengths on one axis, effects are divided by the metric's standard deviation across shots, and the intervals come from a bootstrap.

In [ ]:
def effect(col, n_boot=4000, seed=0):
    g = exp.loc[exp["goal"], col].dropna().to_numpy(dtype=float)
    n = exp.loc[~exp["goal"], col].dropna().to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    boot = (rng.choice(g, (n_boot, len(g))).mean(axis=1)
            - rng.choice(n, (n_boot, len(n))).mean(axis=1))
    sd = exp[col].std()
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"raw_diff": g.mean() - n.mean(), "effect_sd": (g.mean() - n.mean()) / sd, "lo": lo / sd, "hi": hi / sd}


naive = pd.DataFrame({m: effect(m) for m in METRICS}).T
adjusted = pd.DataFrame({m: effect(m + "_adj") for m in METRICS}).T

pd.concat({"shooter only": naive, "minus teammates": adjusted}, axis=1).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ypos = np.arange(len(METRICS))

for tbl, dy, colour, label in [(naive, 0.15, "tab:grey", "shooter only"),
                               (adjusted, -0.15, "tab:blue", "minus teammates")]:
    ax.errorbar(tbl["effect_sd"], ypos + dy,
                xerr=[tbl["effect_sd"] - tbl["lo"], tbl["hi"] - tbl["effect_sd"]],
                fmt="o", color=colour, capsize=3, label=label)

ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(ypos)
ax.set_yticklabels(METRICS)
ax.invert_yaxis()
ax.set_xlabel("change after a goal vs. after a non-goal (in SDs, 95% bootstrap interval)")
ax.set_title(f"What moves in the {W} minutes after scoring?")
ax.legend()
plt.tight_layout()
plt.show()

## 7 · Reading it, and what to do next

**Expect wide intervals.** One tournament only leaves you somewhere around a hundred goals once the window filter is applied (the cell above printed the exact count). Most intervals will straddle zero. That's the honest result at this sample size, and the main lesson: the design is fine, the data is too small. Don't read a story into whichever dot happens to sit furthest from zero.

**Compare the grey and blue dots.** Where they differ, the "effect" was really the whole team changing after a goal (sitting deeper, fewer passes) and not the scorer. This gap is why the teammate subtraction is there.

**The coin isn't fair.** Run the cell below: shots that went in had higher xG than shots that didn't, so the two groups weren't taking the same kind of shot. That needs fixing before any claim.

In [ ]:
exp.groupby("goal")["xg"].describe().round(3)

### Exercises (each one is a real step of the project)

1. **Balance the coin.** Restrict to a band like `0.05 < xg < 0.40`, or better, weight non-goals by `xg / (1 - xg)` so the two groups have the same xG distribution. Does anything change?
2. **Try the window.** Rerun with `W = 10` and `W = 20`. An effect that only exists at one window length isn't an effect.
3. **Fix the bootstrap.** Shots from the same match (and the same player) aren't independent. Resample *matches* instead of shots and see how much the intervals widen.
4. **Negative shocks.** Swap the trigger: a missed big chance (`xg > 0.3`, no goal), a `Dispossessed` or `Miscontrol` in his own half, a `Dribbled Past`. Is the response to bad events bigger than to good ones?
5. **Scale up.** Change `find_season(...)` to one of the full league seasons listed in section 1 and rerun everything. Nothing else in the notebook needs to change, and the intervals should shrink a lot.
6. **Who responds?** With league data, split the effect by position group. Do forwards react differently from midfielders?

### Three ways to define "confidence", in order of ambition

- **Shocks within a match** (this notebook): goal, miss, error. Cleanest causally, shortest horizon.
- **Form across matches**: rolling goals-minus-xG or finishing drought length over a player's last *k* matches, predicting next-match *choices* (shots, xG per shot, take-ons). Needs full seasons, and needs care with regression to the mean.
- **A latent state**: treat confidence as a hidden variable that several observed metrics load on, and fit a hidden Markov or state-space model per player. The estimated loadings answer "which attributes are most affected" directly.

The functions in this notebook now live in `src/confidence_shocks/` (`prep.py`, `features.py`, `shocks.py`, `estimate.py`). New notebooks should import them from there.